In [43]:
import pandas as pd
from functions import calculate_scf, calibrated_count, adjusted_count
df = pd.read_csv("beads_data.csv") #importing csv file in
df = df.rename(columns={"SCF_bead": "scf_count"}) #renaming column to avoid confusion with variable later on
df["scf_count"] = df["scf_count"].astype(str).str.replace(",", "").str.strip()
df["scf_count"] = pd.to_numeric(df["scf_count"], errors="coerce")
df.head() #checking import worked by looking at first first 5 rows

,group,bead_size,bead_number,bead_id,experiment,e_factor,beam_energy,dose_level,exp_variable,scf_count,raw_bead
0,1,1.0,1,1_1,linearity,0.995,6.0,10.0,10.0,40.85,0.8636
1,1,1.0,2,1_2,linearity,0.995,6.0,10.0,10.0,40.84,0.8628
2,1,1.0,3,1_3,linearity,0.995,6.0,10.0,10.0,40.75,0.9771
3,1,1.0,4,1_4,linearity,0.995,6.0,10.0,10.0,42.97,0.8508
4,1,1.0,5,1_5,linearity,0.995,6.0,10.0,10.0,43.52,0.9164


In [ ]:
#data in the table that doesnt have both and experimental value and an scf value is removed
print("Rows before:", len(df)) #number of rows before 
df = df.dropna(subset=["exp_variable"]) #rows with no experimental variable value are removed
print("Rows after:", len(df)) #number of rows after to verify removal
print(df["exp_variable"].isna().sum()) #should show 0 if no rows with missing values remain
df.to_csv("cleaned_beads_data.csv", index=False) #new beads_data file with unwanted values removed, saved to file

Rows before: 600
Rows after: 465
0


In [45]:
#mean raw count of SCF control DM for each set of the beads for use in equations
#mean raw count for set 1 of beads (1-1 to 15-10)
mrc_SCF_control_set1 = (0.2006 + 0.1817 +0.189 + 0.2354 + 0.1846 + 0.1819)/6

#mean raw count for set 2 of the beads (16-1 to 30-10)
mrc_SCF_control_set2 = (0.2659 + 0.2321 + 0.2166 + 0.2513 + 0.229 + 0.2393)/6


In [46]:
#in the cvs data is split into groups 1-4
#scf calculations will occur seperately for each group and will be added into a dataframe
#creating a dictionary to keep track of each group
mean_raw_dict = {
    '1': mrc_SCF_control_set1,
    '2': mrc_SCF_control_set2,
    '3': mrc_SCF_control_set1,
    '4': mrc_SCF_control_set2
}
df['group'] = df['group'].astype(str) #setting group data type to be correct

In [47]:
#creating new column in dataframe to fill with scf for each bead
df['scf_bead'] = pd.NA  

for group_name in df['group'].unique(): #chosing only correct group of beads
    group_df = df[df['group'] == group_name]
    mean_raw = mean_raw_dict[group_name] #setting correct mean raw value
    mean_scf_group = group_df['scf_count'].mean() #finding mean scf count for each group
    scf_values = (mean_scf_group - mean_raw) / (group_df['scf_count'] - mean_raw) #calculating scf
    df.loc[df['group'] == group_name, 'scf_bead'] = scf_values



df.to_csv("beads_data_with_scf.csv", index=False) #saving new dataframe with scf values to csv file
df.head() #checking new dataframe with scf values added in


,group,bead_size,bead_number,bead_id,experiment,e_factor,beam_energy,dose_level,exp_variable,scf_count,raw_bead,scf_bead
0,1,1.0,1,1_1,linearity,0.995,6.0,10.0,10.0,40.85,0.8636,0.976572
1,1,1.0,2,1_2,linearity,0.995,6.0,10.0,10.0,40.84,0.8628,0.976812
2,1,1.0,3,1_3,linearity,0.995,6.0,10.0,10.0,40.75,0.9771,0.97898
3,1,1.0,4,1_4,linearity,0.995,6.0,10.0,10.0,42.97,0.8508,0.928171
4,1,1.0,5,1_5,linearity,0.995,6.0,10.0,10.0,43.52,0.9164,0.916388


In [48]:
#next calculating calibrated counts using the scf values and raw bead counts
df['raw_bead'] = pd.to_numeric(df['raw_bead'], errors='coerce') #data needs to be classed as numeric
df['calibrated_count'] = df['scf_bead'] * df['raw_bead'] #calculating calibrated counts
df.to_csv("beads_data_with_calibrated_counts.csv", index=False) #saving new dataframe with calibrated counts to csv file
df.head()

,group,bead_size,bead_number,bead_id,experiment,e_factor,beam_energy,dose_level,exp_variable,scf_count,raw_bead,scf_bead,calibrated_count
0,1,1.0,1,1_1,linearity,0.995,6.0,10.0,10.0,40.85,0.8636,0.976572,0.843367
1,1,1.0,2,1_2,linearity,0.995,6.0,10.0,10.0,40.84,0.8628,0.976812,0.842793
2,1,1.0,3,1_3,linearity,0.995,6.0,10.0,10.0,40.75,0.9771,0.97898,0.956561
3,1,1.0,4,1_4,linearity,0.995,6.0,10.0,10.0,42.97,0.8508,0.928171,0.789688
4,1,1.0,5,1_5,linearity,0.995,6.0,10.0,10.0,43.52,0.9164,0.916388,0.839778


In [49]:
#creating controls column to be used in adjusted count calculation
# True if bead is a control (exp_variable == 0), if not, False
df['is_control'] = df['exp_variable'] == 0

# Mean calibrated count of controls per group
mean_control = df[df['is_control']].groupby('group')['calibrated_count'].mean()
df['mean_control'] = df['group'].map(mean_control) #adding into df
df.head()

,group,bead_size,bead_number,bead_id,experiment,e_factor,beam_energy,dose_level,exp_variable,scf_count,raw_bead,scf_bead,calibrated_count,is_control,mean_control
0,1,1.0,1,1_1,linearity,0.995,6.0,10.0,10.0,40.85,0.8636,0.976572,0.843367,False,0.611921
1,1,1.0,2,1_2,linearity,0.995,6.0,10.0,10.0,40.84,0.8628,0.976812,0.842793,False,0.611921
2,1,1.0,3,1_3,linearity,0.995,6.0,10.0,10.0,40.75,0.9771,0.97898,0.956561,False,0.611921
3,1,1.0,4,1_4,linearity,0.995,6.0,10.0,10.0,42.97,0.8508,0.928171,0.789688,False,0.611921
4,1,1.0,5,1_5,linearity,0.995,6.0,10.0,10.0,43.52,0.9164,0.916388,0.839778,False,0.611921


In [50]:
#calulating adjusted counts using the mean control values and calibrated counts
df['adjusted_count'] = df['scf_bead'] * (df['raw_bead'] - df['mean_control'])
df.to_csv("beads_data_with_adjusted_counts.csv", index=False) #saving new dataframe with adjusted counts to csv file
df.head()

,group,bead_size,bead_number,bead_id,experiment,e_factor,beam_energy,dose_level,exp_variable,scf_count,raw_bead,scf_bead,calibrated_count,is_control,mean_control,adjusted_count
0,1,1.0,1,1_1,linearity,0.995,6.0,10.0,10.0,40.85,0.8636,0.976572,0.843367,False,0.611921,0.245783
1,1,1.0,2,1_2,linearity,0.995,6.0,10.0,10.0,40.84,0.8628,0.976812,0.842793,False,0.611921,0.245062
2,1,1.0,3,1_3,linearity,0.995,6.0,10.0,10.0,40.75,0.9771,0.97898,0.956561,False,0.611921,0.357503
3,1,1.0,4,1_4,linearity,0.995,6.0,10.0,10.0,42.97,0.8508,0.928171,0.789688,False,0.611921,0.22172
4,1,1.0,5,1_5,linearity,0.995,6.0,10.0,10.0,43.52,0.9164,0.916388,0.839778,False,0.611921,0.279021


In [53]:
# keeping columns numeric 
df['adjusted_count'] = pd.to_numeric(df['adjusted_count'], errors='coerce')
df['raw_bead'] = pd.to_numeric(df['raw_bead'], errors='coerce')
df['e_factor'] = pd.to_numeric(df['e_factor'], errors='coerce')

# Calculate dose
df['dose'] = df['adjusted_count'] * (df['raw_bead'] / df['e_factor'])
df.to_csv("beads_data_with_dose.csv", index=False) #saving new dataframe with dose to csv file
df.head()

,group,bead_size,bead_number,bead_id,experiment,e_factor,beam_energy,dose_level,exp_variable,scf_count,raw_bead,scf_bead,calibrated_count,is_control,mean_control,adjusted_count,dose
0,1,1.0,1,1_1,linearity,0.995,6.0,10.0,10.0,40.85,0.8636,0.976572,0.843367,False,0.611921,0.245783,0.213324
1,1,1.0,2,1_2,linearity,0.995,6.0,10.0,10.0,40.84,0.8628,0.976812,0.842793,False,0.611921,0.245062,0.212502
2,1,1.0,3,1_3,linearity,0.995,6.0,10.0,10.0,40.75,0.9771,0.97898,0.956561,False,0.611921,0.357503,0.351071
3,1,1.0,4,1_4,linearity,0.995,6.0,10.0,10.0,42.97,0.8508,0.928171,0.789688,False,0.611921,0.221720,0.189588
4,1,1.0,5,1_5,linearity,0.995,6.0,10.0,10.0,43.52,0.9164,0.916388,0.839778,False,0.611921,0.279021,0.256980
